# 🏃 Personal Running Coach Agent Swarm
## Phase 3 — ChromaDB Memory Ingestion

**Project:** Agentic AI Course — Will Sutherland, May 2026

Builds and populates the three ChromaDB collections that all agents
will query for context during Phase 4.

### Collections

| Collection | Content | Updated |
|---|---|---|
| `workout_summaries` | Natural language summary of every run from the last 6 months | Re-run after each Strava/Garmin refresh |
| `athlete_profile` | Static athlete background, goals, training context | Edit manually when profile changes |
| `session_notes` | Free-text notes added after key sessions | Add via the notes cell at the bottom |

### This notebook covers
1. Setup — mount Drive, imports, connect to ChromaDB
2. Build `workout_summaries` collection
3. Build `athlete_profile` collection
4. Build `session_notes` collection
5. Build retrieval wrapper used by all agents
6. Smoke tests — verify retrieval is working
7. Add session notes (run this cell after key workouts)

---
## 1. Setup

In [9]:
import os, sys, json, sqlite3
from datetime import date, timedelta
from google.colab import drive, userdata

drive.mount('/content/drive', force_remount=False)

BASE_DIR       = "/content/drive/MyDrive/running_coach"
WORKOUTS_PATH  = f"{BASE_DIR}/data/processed/workouts_normalized.json"
GARMIN_DB_PATH = f"{BASE_DIR}/data/raw/garmin/garmin.db"
CHROMA_DIR     = f"{BASE_DIR}/memory/chroma"

sys.path.insert(0, f"{BASE_DIR}/tools")

GEMINI_API_KEY = userdata.get('key')

%pip install -q chromadb google-generativeai

import chromadb
from chromadb.utils import embedding_functions
import google.generativeai as genai

genai.configure(api_key=GEMINI_API_KEY)

# ── ChromaDB client (persisted to Drive) ──────────────────────────────────────
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)

# ── Embedding function — Gemini gemini-embedding-001 ────────────────────────
gemini_ef = embedding_functions.GoogleGenerativeAiEmbeddingFunction(
    api_key=GEMINI_API_KEY,
    model_name="models/gemini-embedding-001",
)

print("✅ Setup complete.")
print(f"   ChromaDB path:    {CHROMA_DIR}")
print(f"   Workouts file:    {os.path.exists(WORKOUTS_PATH)}")
print(f"   Garmin DB:        {os.path.exists(GARMIN_DB_PATH)}")
print(f"   Existing collections: {[c.name for c in chroma_client.list_collections()]}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Setup complete.
   ChromaDB path:    /content/drive/MyDrive/running_coach/memory/chroma
   Workouts file:    True
   Garmin DB:        True
   Existing collections: ['workout_summaries']


---
## 2. `workout_summaries` Collection

Each WorkoutRecord is converted to a natural language summary chunk and
embedded into ChromaDB. Agents retrieve the most semantically relevant
past sessions when answering questions about training history.

**Chunk format:**
```
Date: 2026-04-22 | Type: threshold | Distance: 12.3km | Duration: 49.5min
Pace: 4:03/km (target: 4:01/km, 2s slower) | HR: 162 avg / 178 max
Training load: 187 | Aerobic effect: 3.8 | Suffer score: 67
Recovery context: HRV 95 (balanced), sleep 7.8h, body battery at wake: 62, readiness: 74
```

> **Safe to re-run** — deletes and rebuilds the collection each time
> so you always have a fresh, consistent index after a data refresh.

In [10]:
from parse_workout_data import load_workouts, format_pace
from calculate_pace_zones import classify_workouts, format_pace

# ── Load and classify workouts ─────────────────────────────────────────────────
workouts  = load_workouts(WORKOUTS_PATH, days=180)
enriched  = classify_workouts(workouts)
print(f"Loaded {len(enriched)} workouts for ingestion.")


def build_workout_chunk(w: dict) -> str:
    """
    Convert a classified WorkoutRecord to a natural language summary
    suitable for semantic embedding and retrieval.
    """
    cl = w.get('classification', {})
    pe = w.get('pace_evaluation')
    hc = w.get('health_context', {})

    workout_type = cl.get('workout_type') or 'unclassified'
    pace_str     = format_pace(w.get('avg_pace_min_km'))

    # Pace evaluation line
    if pe and pe.get('verdict'):
        pace_line = f"Pace: {pace_str} | {pe['verdict']}"
    else:
        pace_line = f"Pace: {pace_str}"

    # Health context line
    hc_parts = []
    if hc.get('hrv_status'):
        hc_parts.append(f"HRV status: {hc['hrv_status']}")
    if hc.get('hrv_last_night'):
        hc_parts.append(f"HRV last night: {hc['hrv_last_night']}")
    if hc.get('sleep_sleep_time_seconds'):
        hrs = round(hc['sleep_sleep_time_seconds'] / 3600, 1)
        hc_parts.append(f"sleep: {hrs}h")
    if hc.get('body_battery_at_wake'):
        hc_parts.append(f"body battery at wake: {hc['body_battery_at_wake']}")
    if hc.get('training_readiness_score'):
        hc_parts.append(f"readiness: {hc['training_readiness_score']}")
    if hc.get('heart_rate_resting_hr'):
        hc_parts.append(f"resting HR: {hc['heart_rate_resting_hr']} bpm")

    hc_line = f"Recovery context: {', '.join(hc_parts)}" if hc_parts else ""

    lines = [
        f"Date: {w.get('date')} | Type: {workout_type} | "
        f"Distance: {w.get('distance_km')}km | Duration: {w.get('duration_min')}min",
        pace_line,
        f"HR: {w.get('avg_hr')} avg / {w.get('max_hr')} max | "
        f"Elevation: {w.get('elevation_m')}m",
        f"Training load: {w.get('training_load')} | "
        f"Aerobic effect: {w.get('aerobic_effect')} | "
        f"Anaerobic effect: {w.get('anaerobic_effect')} | "
        f"Suffer score: {w.get('suffer_score')}",
    ]
    if hc_line:
        lines.append(hc_line)
    if w.get('garmin_enriched'):
        lines.append("Garmin enriched: yes")

    return "\n".join(lines)


# ── Build collection ───────────────────────────────────────────────────────────
# Delete and recreate for a clean rebuild
try:
    chroma_client.delete_collection("workout_summaries")
    print("Deleted existing workout_summaries collection.")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="workout_summaries",
    embedding_function=gemini_ef,
    metadata={"hnsw:space": "cosine"}
)

# Ingest in batches of 10 (Gemini embedding API rate limit)
import time
BATCH_SIZE = 10
documents, ids, metadatas = [], [], []

for w in enriched:
    chunk = build_workout_chunk(w)
    documents.append(chunk)
    ids.append(w['activity_id'])
    metadatas.append({
        'date':         w.get('date', ''),
        'workout_type': w.get('classification', {}).get('workout_type') or 'unknown',
        'distance_km':  str(w.get('distance_km', '')),
        'garmin_enriched': str(w.get('garmin_enriched', False)),
        'source':       w.get('source', ''),
    })

total    = len(documents)
ingested = 0
for i in range(0, total, BATCH_SIZE):
    batch_docs  = documents[i:i+BATCH_SIZE]
    batch_ids   = ids[i:i+BATCH_SIZE]
    batch_meta  = metadatas[i:i+BATCH_SIZE]
    collection.add(documents=batch_docs, ids=batch_ids, metadatas=batch_meta)
    ingested += len(batch_docs)
    print(f"  Ingested {ingested}/{total}...")
    time.sleep(1)  # Respect Gemini API rate limit

print(f"\n✅ workout_summaries: {collection.count()} documents ingested.")

Loaded 115 workouts for ingestion.
Deleted existing workout_summaries collection.
  Ingested 10/115...
  Ingested 20/115...
  Ingested 30/115...
  Ingested 40/115...
  Ingested 50/115...
  Ingested 60/115...
  Ingested 70/115...
  Ingested 80/115...
  Ingested 90/115...
  Ingested 100/115...
  Ingested 110/115...
  Ingested 115/115...

✅ workout_summaries: 115 documents ingested.


---
## 3. `athlete_profile` Collection

Static document describing Will's training background, current block,
goals, and coaching context. Agents retrieve this for personalisation.
Edit the profile dict below when anything changes (e.g. after the race).

> **Safe to re-run** — rebuilds the profile document each time.

In [11]:
# ── Athlete profile document ───────────────────────────────────────────────────
# Edit this dict to update the profile. Re-run the cell to push changes.

ATHLETE_PROFILE = {
    "name":             "Will Sutherland",
    "location":         "Halifax, Nova Scotia",
    "running_since":    "Fall 2023",
    "experience":       "Approximately 2.5 years of serious training",

    "race_history": [
        {"event": "Half Marathon", "time": "1:27:50"},
        {"event": "Half Marathon", "time": "1:28:xx"},
        {"event": "Full Marathon", "time": "3:09", "notes": "Strong race"},
        {"event": "Full Marathon", "time": "3:19",
         "notes": "Injured, walked last 12km. Was on 4:15/km pace through 30km."},
    ],

    "current_race": {
        "name":    "Fredericton Half Marathon",
        "date":    "2026-05-10",
        "goal_a":  "1:23:59 (A+ goal)",
        "goal_b":  "Sub 1:25:00 (happy with this)",
        "race_strategy": "Negative split — conservative first half, build in second half",
    },

    "current_block": {
        "plan_length":    "18 weeks total (16 weeks build + 2 week taper)",
        "training_days":  "5 days per week: Tuesday, Wednesday, Thursday, Saturday, Sunday",
        "rest_days":      "Monday and Friday",
        "typical_week": [
            {"day": "Tuesday",   "session": "Easy run or fartlek"},
            {"day": "Wednesday", "session": "Speed workout (key session) — e.g. 10x1km at HM pace"},
            {"day": "Thursday",  "session": "Easy run"},
            {"day": "Saturday",  "session": "Long run with structure — warm up/cool down at MP+10s, middle at HM pace"},
            {"day": "Sunday",    "session": "Easy run"},
        ],
    },

    "training_paces": {
        "easy":      "By feel — anything slower than 4:45/km",
        "marathon":  "4:14/km",
        "threshold": "4:01/km",
        "1hr":       "3:56/km",
        "fartlek":   "3:49/km",
        "8k":        "3:46/km",
        "vo2max":    "3:40/km",
        "hm_race":   "~3:59/km (for 1:24 target)",
    },

    "devices":          "Garmin Forerunner 265",
    "data_sources":     "Garmin Connect (via garmin-givemydata), Strava",

    "injuries": {
        "current":   "None",
        "history":   "Previous marathon DNF-equivalent due to injury at 30km mark",
        "watch_areas": "Monitor for general overtraining signs given training volume",
    },

    "coaching_context": (
        "Will trains under a coach on a structured 18-week periodized plan. "
        "The Wednesday speed session and Saturday long run are the key quality sessions each week. "
        "Easy runs should be genuinely easy — by feel, no pace target, ceiling at 4:45/km. "
        "The 10% mileage rule applies week-on-week. "
        "Injury disclaimer: any injury-related queries should be redirected to Will's coach or a physio."
    ),

    "agent_notes": (
        "When evaluating workouts, always compare against coach-prescribed targets, not generic benchmarks. "
        "Pace tolerance is +/- 5 seconds from target. "
        "Recovery decisions should weight: training readiness score, HRV status, body battery at wake, and TSB. "
        "Race predictions from Garmin suggest ~1:23 half marathon fitness — consistent with goal."
    ),
}


def profile_to_text(profile: dict) -> str:
    """Convert the athlete profile dict to a flat text document for embedding."""
    lines = [
        f"Athlete: {profile['name']}, {profile['location']}",
        f"Running since: {profile['running_since']} ({profile['experience']})",
        "",
        "Race history:",
    ]
    for r in profile['race_history']:
        note = f" ({r['notes']})" if r.get('notes') else ""
        lines.append(f"  {r['event']}: {r['time']}{note}")

    cr = profile['current_race']
    lines += [
        "",
        f"Target race: {cr['name']} on {cr['date']}",
        f"Goal A: {cr['goal_a']}",
        f"Goal B: {cr['goal_b']}",
        f"Race strategy: {cr['race_strategy']}",
        "",
        f"Training plan: {profile['current_block']['plan_length']}",
        f"Training days: {profile['current_block']['training_days']}",
        f"Rest days: {profile['current_block']['rest_days']}",
        "",
        "Typical weekly structure:",
    ]
    for day in profile['current_block']['typical_week']:
        lines.append(f"  {day['day']}: {day['session']}")

    lines += [
        "",
        "Training paces (coach-prescribed):",
    ]
    for pace_type, pace in profile['training_paces'].items():
        lines.append(f"  {pace_type}: {pace}")

    inj = profile['injuries']
    lines += [
        "",
        f"Current injuries: {inj['current']}",
        f"Injury history: {inj['history']}",
        f"Watch areas: {inj['watch_areas']}",
        "",
        f"Coaching context: {profile['coaching_context']}",
        "",
        f"Agent notes: {profile['agent_notes']}",
    ]
    return "\n".join(lines)


profile_text = profile_to_text(ATHLETE_PROFILE)

# ── Build collection ───────────────────────────────────────────────────────────
try:
    chroma_client.delete_collection("athlete_profile")
except Exception:
    pass

profile_col = chroma_client.create_collection(
    name="athlete_profile",
    embedding_function=gemini_ef,
    metadata={"hnsw:space": "cosine"}
)

# Split into paragraphs for finer-grained retrieval
paragraphs = [p.strip() for p in profile_text.split('\n\n') if p.strip()]
profile_col.add(
    documents=paragraphs,
    ids=[f"profile_{i}" for i in range(len(paragraphs))],
    metadatas=[{"section": "athlete_profile", "chunk": i} for i in range(len(paragraphs))]
)

print(f"✅ athlete_profile: {profile_col.count()} chunks ingested.")
print("\nProfile document:")
print(profile_text)

✅ athlete_profile: 9 chunks ingested.

Profile document:
Athlete: Will Sutherland, Halifax, Nova Scotia
Running since: Fall 2023 (Approximately 2.5 years of serious training)

Race history:
  Half Marathon: 1:27:50
  Half Marathon: 1:28:xx
  Full Marathon: 3:09 (Strong race)
  Full Marathon: 3:19 (Injured, walked last 12km. Was on 4:15/km pace through 30km.)

Target race: Fredericton Half Marathon on 2026-05-10
Goal A: 1:23:59 (A+ goal)
Goal B: Sub 1:25:00 (happy with this)
Race strategy: Negative split — conservative first half, build in second half

Training plan: 18 weeks total (16 weeks build + 2 week taper)
Training days: 5 days per week: Tuesday, Wednesday, Thursday, Saturday, Sunday
Rest days: Monday and Friday

Typical weekly structure:
  Tuesday: Easy run or fartlek
  Wednesday: Speed workout (key session) — e.g. 10x1km at HM pace
  Thursday: Easy run
  Saturday: Long run with structure — warm up/cool down at MP+10s, middle at HM pace
  Sunday: Easy run

Training paces (coach-

---
## 4. `session_notes` Collection

Free-text notes added after key sessions. These give agents qualitative
context alongside the numbers — how the run felt, any niggles, weather,
mental state, etc.

**To add a note:** scroll to Section 7 at the bottom and fill in the
date and note text, then run that cell.

This cell creates the collection if it doesn't exist. It does **not**
wipe existing notes — notes accumulate over time.

In [12]:
# Get or create — does NOT delete existing notes
notes_col = chroma_client.get_or_create_collection(
    name="session_notes",
    embedding_function=gemini_ef,
    metadata={"hnsw:space": "cosine"}
)

print(f"✅ session_notes collection ready: {notes_col.count()} note(s) stored.")
if notes_col.count() > 0:
    existing = notes_col.get()
    print("\nExisting notes:")
    for doc, meta in zip(existing['documents'], existing['metadatas']):
        print(f"  [{meta.get('date', 'unknown')}] {doc[:120]}{'...' if len(doc) > 120 else ''}")

✅ session_notes collection ready: 0 note(s) stored.


---
## 5. Retrieval Wrapper

A single function that all agents call to query memory.
Writes `memory_retrieval.py` to `tools/` on Drive.

In [17]:
memory_retrieval_code = (
    '"""\n'
    'tools/memory_retrieval.py\n\n'
    'Retrieval wrapper for the ChromaDB vector store.\n'
    'All agents call these functions — no agent queries ChromaDB directly.\n'
    '"""\n\n'
    'from __future__ import annotations\n'
    'import chromadb\n'
    'from chromadb.utils import embedding_functions\n\n\n'
    'def get_client(chroma_dir: str, api_key: str) -> tuple:\n'
    '    client = chromadb.PersistentClient(path=chroma_dir)\n'
    '    ef = embedding_functions.GoogleGenerativeAiEmbeddingFunction(\n'
    '        api_key=api_key,\n'
    '        model_name="models/gemini-embedding-001",\n'
    '    )\n'
    '    return client, ef\n\n\n'
    'def retrieve_workouts(client, ef, query: str, n: int = 5, filters: dict | None = None) -> list[dict]:\n'
    '    try:\n'
    '        col = client.get_collection("workout_summaries", embedding_function=ef)\n'
    '    except Exception:\n'
    '        return []\n'
    '    kwargs = {"query_texts": [query], "n_results": min(n, col.count())}\n'
    '    if filters:\n'
    '        kwargs["where"] = filters\n'
    '    results = col.query(**kwargs)\n'
    '    return [\n'
    '        {"document": doc, "metadata": meta, "distance": dist}\n'
    '        for doc, meta, dist in zip(\n'
    '            results["documents"][0], results["metadatas"][0], results["distances"][0]\n'
    '        )\n'
    '    ]\n\n\n'
    'def retrieve_profile(client, ef, query: str = "athlete background goals training context", n: int = 5) -> str:\n'
    '    try:\n'
    '        col = client.get_collection("athlete_profile", embedding_function=ef)\n'
    '    except Exception:\n'
    '        return ""\n'
    '    results = col.query(query_texts=[query], n_results=min(n, col.count()))\n'
    '    return "\\n\\n".join(results["documents"][0])\n\n\n'
    'def retrieve_notes(client, ef, query: str, n: int = 3) -> list[dict]:\n'
    '    try:\n'
    '        col = client.get_collection("session_notes", embedding_function=ef)\n'
    '        if col.count() == 0:\n'
    '            return []\n'
    '    except Exception:\n'
    '        return []\n'
    '    results = col.query(query_texts=[query], n_results=min(n, col.count()))\n'
    '    return [\n'
    '        {"document": doc, "metadata": meta, "distance": dist}\n'
    '        for doc, meta, dist in zip(\n'
    '            results["documents"][0], results["metadatas"][0], results["distances"][0]\n'
    '        )\n'
    '    ]\n\n\n'
    'def retrieve_all(client, ef, query: str, n_workouts: int = 5, n_profile: int = 4, n_notes: int = 3) -> dict:\n'
    '    return {\n'
    '        "workouts": retrieve_workouts(client, ef, query, n=n_workouts),\n'
    '        "profile":  retrieve_profile(client, ef, query, n=n_profile),\n'
    '        "notes":    retrieve_notes(client, ef, query, n=n_notes),\n'
    '    }\n\n\n'
    'def format_memory_context(results: dict) -> str:\n'
    '    lines = ["=== Memory Context ==="]\n'
    '    if results.get("profile"):\n'
    '        lines += ["\\n--- Athlete Profile ---", results["profile"]]\n'
    '    if results.get("workouts"):\n'
    '        lines.append("\\n--- Relevant Past Workouts ---")\n'
    '        for r in results["workouts"]:\n'
    '            lines.append(r["document"])\n'
    '            lines.append("")\n'
    '    if results.get("notes"):\n'
    '        lines.append("--- Session Notes ---")\n'
    '        for r in results["notes"]:\n'
    '            d = r["metadata"].get("date", "unknown")\n'
    '            lines.append(f"[{d}] {r[\'document\']}")\n'
    '    return "\\n".join(lines)\n'
)

tool_path = os.path.join(BASE_DIR, "tools", "memory_retrieval.py")
with open(tool_path, 'w') as f:
    f.write(memory_retrieval_code)

print(f"✅ memory_retrieval.py written to {tool_path}")

# Verify it's importable
import importlib, sys
if 'memory_retrieval' in sys.modules:
    del sys.modules['memory_retrieval']
import memory_retrieval
print("✅ Import successful")

✅ memory_retrieval.py written to /content/drive/MyDrive/running_coach/tools/memory_retrieval.py
✅ Import successful


---
## 6. Smoke Tests

In [18]:
print("=" * 60)
print("TEST 6.1 — Collection counts")
print("=" * 60)
for name in ['workout_summaries', 'athlete_profile', 'session_notes']:
    try:
        col   = chroma_client.get_collection(name, embedding_function=gemini_ef)
        count = col.count()
        status = '✅' if count > 0 else '⚠️  empty'
        print(f"  {status} {name}: {count} documents")
    except Exception as e:
        print(f"  ❌ {name}: {e}")

TEST 6.1 — Collection counts
  ✅ workout_summaries: 115 documents
  ✅ athlete_profile: 9 documents
  ⚠️  empty session_notes: 0 documents


In [19]:
print("=" * 60)
print("TEST 6.2 — Workout retrieval")
print("=" * 60)

from memory_retrieval import get_client, retrieve_workouts, retrieve_profile, retrieve_all, format_memory_context

mem_client, mem_ef = get_client(CHROMA_DIR, GEMINI_API_KEY)

test_queries = [
    "recent threshold sessions",
    "long run at marathon pace",
    "easy recovery run",
]

for query in test_queries:
    results = retrieve_workouts(mem_client, mem_ef, query, n=2)
    print(f"\nQuery: '{query}' → {len(results)} results")
    for r in results:
        first_line = r['document'].split('\n')[0]
        print(f"  [{r['metadata'].get('date')}] {first_line} (dist: {r['distance']:.3f})")

TEST 6.2 — Workout retrieval

Query: 'recent threshold sessions' → 2 results
  [2026-02-20] Date: 2026-02-20 | Type: unclassified | Distance: 2.3km | Duration: 12.0min (dist: 0.279)
  [2025-12-10] Date: 2025-12-10 | Type: unclassified | Distance: 10.36km | Duration: 45.25min (dist: 0.286)

Query: 'long run at marathon pace' → 2 results
  [2025-11-08] Date: 2025-11-08 | Type: unclassified | Distance: 42.5km | Duration: 199.53min (dist: 0.175)
  [2026-04-08] Date: 2026-04-08 | Type: vo2max | Distance: 14.48km | Duration: 66.4min (dist: 0.183)

Query: 'easy recovery run' → 2 results
  [2026-01-13] Date: 2026-01-13 | Type: easy | Distance: 8.33km | Duration: 43.28min (dist: 0.146)
  [2026-04-11] Date: 2026-04-11 | Type: easy | Distance: 4.0km | Duration: 20.0min (dist: 0.146)


In [20]:
print("=" * 60)
print("TEST 6.3 — Profile retrieval")
print("=" * 60)

profile_result = retrieve_profile(
    mem_client, mem_ef,
    query="athlete goals race target pace strategy",
    n=3
)
print(profile_result)
print("\n✅ Profile retrieval working.")

TEST 6.3 — Profile retrieval
Target race: Fredericton Half Marathon on 2026-05-10
Goal A: 1:23:59 (A+ goal)
Goal B: Sub 1:25:00 (happy with this)
Race strategy: Negative split — conservative first half, build in second half

Training paces (coach-prescribed):
  easy: By feel — anything slower than 4:45/km
  marathon: 4:14/km
  threshold: 4:01/km
  1hr: 3:56/km
  fartlek: 3:49/km
  8k: 3:46/km
  vo2max: 3:40/km
  hm_race: ~3:59/km (for 1:24 target)

Agent notes: When evaluating workouts, always compare against coach-prescribed targets, not generic benchmarks. Pace tolerance is +/- 5 seconds from target. Recovery decisions should weight: training readiness score, HRV status, body battery at wake, and TSB. Race predictions from Garmin suggest ~1:23 half marathon fitness — consistent with goal.

✅ Profile retrieval working.


In [21]:
print("=" * 60)
print("TEST 6.4 — Full memory context (as agents will see it)")
print("=" * 60)

results = retrieve_all(
    mem_client, mem_ef,
    query="how did my last threshold session go?",
    n_workouts=3,
    n_profile=2,
    n_notes=2,
)
context = format_memory_context(results)
print(context)
print("\n✅ Full memory context assembled correctly.")

TEST 6.4 — Full memory context (as agents will see it)
=== Memory Context ===

--- Athlete Profile ---
Agent notes: When evaluating workouts, always compare against coach-prescribed targets, not generic benchmarks. Pace tolerance is +/- 5 seconds from target. Recovery decisions should weight: training readiness score, HRV status, body battery at wake, and TSB. Race predictions from Garmin suggest ~1:23 half marathon fitness — consistent with goal.

Training paces (coach-prescribed):
  easy: By feel — anything slower than 4:45/km
  marathon: 4:14/km
  threshold: 4:01/km
  1hr: 3:56/km
  fartlek: 3:49/km
  8k: 3:46/km
  vo2max: 3:40/km
  hm_race: ~3:59/km (for 1:24 target)

--- Relevant Past Workouts ---
Date: 2026-04-08 | Type: vo2max | Distance: 14.48km | Duration: 66.4min
Pace: 4:35/km | Vo2max target: 3:40/km. Actual: 4:35/km (55s slower than target). Slightly under — check conditions, fatigue, or HR data. ⚠️
HR: 146.8 avg / 174 max | Elevation: 107.0m
Training load: 319.214660644531

---
## 7. Add Session Notes

Run this cell after any key session to add a qualitative note.
Notes are stored permanently in ChromaDB and retrieved by agents
when relevant to a query.

**Tips for good notes:**
- How did the effort feel vs the pace target?
- Any physical feedback (legs, breathing, niggles)?
- External factors (weather, sleep the night before, stress)?
- Did you hit the workout as prescribed or modify it?

In [22]:
# ── Edit these two fields and run the cell ─────────────────────────────────────
NOTE_DATE = "2026-05-02"   # YYYY-MM-DD — date of the session
NOTE_TEXT = """            # Your note here — no length limit
Last Saturday long run before Freddy Half. 18mins at 1HR pace, 2mins recovery jog and then 5mins at Fartlek.
Good workout - felt a little difficult but also went faster than prescribed paces. Finish the Fartlek strong
and am pumped for Freddy!
"""
# ─────────────────────────────────────────────────────────────────────────────

note_text = NOTE_TEXT.strip()
if not note_text or note_text.startswith("Replace this"):
    print("⚠️  Please edit NOTE_DATE and NOTE_TEXT before running this cell.")
else:
    note_id = f"note_{NOTE_DATE}_{abs(hash(note_text)) % 10000:04d}"
    notes_col = chroma_client.get_or_create_collection(
        name="session_notes", embedding_function=gemini_ef
    )
    notes_col.add(
        documents=[note_text],
        ids=[note_id],
        metadatas=[{"date": NOTE_DATE, "added": str(date.today())}]
    )
    print(f"✅ Note added for {NOTE_DATE} (id: {note_id})")
    print(f"   Total notes in collection: {notes_col.count()}")
    print(f"\nNote text:\n{note_text}")

✅ Note added for 2026-05-02 (id: note_2026-05-02_9765)
   Total notes in collection: 1

Note text:
# Your note here — no length limit
Last Saturday long run before Freddy Half. 18mins at 1HR pace, 2mins recovery jog and then 5mins at Fartlek. 
Good workout - felt a little difficult but also went faster than prescribed paces. Finish the Fartlek strong
and am pumped for Freddy!


---
## ✅ Phase 3 Complete

**What's now in place:**
- `workout_summaries` — all runs from last 6 months embedded and queryable
- `athlete_profile` — Will's background, goals, paces, coaching context embedded
- `session_notes` — persistent notes collection ready to receive post-session entries
- `memory_retrieval.py` — retrieval wrapper written to `tools/` on Drive

**Ongoing workflow:**

| Action | What to do |
|---|---|
| After a Strava/Garmin data refresh | Re-run Section 2 to rebuild workout_summaries |
| After a key session | Fill in Section 7 and run the add-note cell |
| Profile change (e.g. post-race) | Edit ATHLETE_PROFILE dict in Section 3, re-run |

**Next — Phase 4: Agents**
- `agents/feedback_agent.py` — pace evaluation, workout analysis vs targets
- `agents/recovery_agent.py` — training load monitoring, recovery advice
- `agents/planner_agent.py` — weekly schedule generation and adjustment
- `agents/coordinator_agent.py` — orchestration, routing, self-critique